# Treinamento ResNet + Lstm Multimodal

Será implementada uma nova loss baseada em BCE e Margin Ranking Loss.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 1. Setup

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import sys
import datetime
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

import optuna
from transformers import AutoModel # transformers==4.44.0
import einops
import timm
import torchaudio # torchaudio==2.5.1

sys.path.insert(0, "/mnt/storage_C4/gaussian_football")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/preprocessing")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/models/nn")

from models.nn.resnetlstm import ResNetLSTM
from models.nn.resnetlstm_multimodal import ResNetLSTM_MultiModal
#from preprocessing.loader_multimodal_pair import train_video_transform, train_mel_transform, TARGET_SHAPE, build_multimodal_dataloader
from preprocessing.loader_multimodal_frac2 import build_multimodal_dataloader, train_video_transform, TARGET_SHAPE, train_mel_transform
from preprocessing.balanced_dataset import balanced_df

/mnt/storage_C4/gaussian_football/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 435
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Torch: 2.5.1+cu121
Torchvision: 0.20.1+cu121
CUDA disponível: True
CUDA: 12.1
GPU: NVIDIA RTX A4500
Device: cuda


## 2. Configuração

In [ ]:
# paths
LABELS_ALL   = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_combinadas_wg.csv"
LABELS_DIR   = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used"
CKPT_DIR     = "/mnt/storage_C4/gaussian_football/models/checkpoints"
RUNS_DIR     = "/mnt/storage_C4/gaussian_football/runs_focused_resnet50_frozen_fusion_bn"  # logs do TensorBoard

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# treino 
EPOCHS        = 100        # teto por causa do early stopping decide
PATIENCE      = 20         # épocas sem melhora antes de parar
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0
MONITOR       = "loss"      # métrica de checkpoint/early-stopping: "ccc" | "mae" | "loss"

# dataloaders (mudei pra 1 pra rodar a resnet 50 c finetune)
BATCH_SIZE = 1
BATCH_SIZE_RES50 = 1

# modelo padrão
MODEL_DEFAULTS = dict(
    frame_step=1,
    hidden_size=256,
    num_layers=1,
    use_dropout=False,
    dropout_p=0.3,
)

## 3. Balanceando os Dados

Gera os CSVs balanceados (50% highlight / 50% não-highlight por partida,
com `threshold=0.5` no `arousal_score`). Só recalcula se os arquivos não existirem
(use `FORCE_REBUILD = True` para refazer).

In [6]:

FORCE_REBUILD = True

paths_balanced = {
    "train": os.path.join(LABELS_DIR, "balanced_todas_as_ligas_train_wg.csv"),
    "valid": os.path.join(LABELS_DIR, "balanced_todas_as_ligas_valid_wg.csv"),
    "test":  os.path.join(LABELS_DIR, "balanced_todas_as_ligas_test_wg.csv"),
}

if FORCE_REBUILD or not all(os.path.exists(p) for p in paths_balanced.values()):
    all_data = pd.read_csv(LABELS_ALL)
    for split, out_path in paths_balanced.items():
        subset = all_data[all_data["split"] == split]
        balanced = balanced_df(subset, "game_id", threshold=0.5, random_state=SEED)
        balanced.to_csv(out_path, index=False)
        print(f"{split}: {len(balanced)} clipes -> {out_path}")
else:
    print("CSVs balanceados já existem. (FORCE_REBUILD=True para refazer.)")

train: 16912 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_train_wg.csv
valid: 6152 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_valid_wg.csv
test: 5730 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_test_wg.csv


## 4. DataLoaders Multimodais

In [7]:
TRAIN_PATH = f"{LABELS_DIR}/todas_as_ligas_train_wg.csv"
VAL_PATH   = f"{LABELS_DIR}/todas_as_ligas_valid_wg.csv"
TEST_PATH   = f"{LABELS_DIR}/todas_as_ligas_test_wg.csv"

In [8]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df['type'].value_counts()

type
Background    56396
shot          47936
goal           8460
Name: count, dtype: int64

Filtrando os datasets apenas por `goal` (high score) e `Background` (low score).

In [9]:
eventos_usados = ['goal', 'Background']
dir_background_goal = '/mnt/storage_C4/gaussian_football/data/datasets_goal_background'

# train
train_filtrado = train_df[train_df['type'].isin(eventos_usados)]
print('train:\n', train_filtrado['type'].value_counts())
train_filtrado.to_csv(f'{dir_background_goal}/train.csv', index=False)

# val
val_filtrado = val_df[val_df['type'].isin(eventos_usados)]
print('\n val:\n', val_filtrado['type'].value_counts())
val_filtrado.to_csv(f'{dir_background_goal}/val.csv', index=False)

# test
test_filtrado = test_df[test_df['type'].isin(eventos_usados)]
print('\n test:\n', test_filtrado['type'].value_counts())
test_filtrado.to_csv(f'{dir_background_goal}/test.csv', index=False)

train:
 type
Background    56396
goal           8460
Name: count, dtype: int64

 val:
 type
Background    17917
goal           3076
Name: count, dtype: int64

 test:
 type
Background    18697
goal           2865
Name: count, dtype: int64


## FIltrado por clipes não corrompidoss

In [10]:
TRAIN_PATH = f'{dir_background_goal}/train_filtered.csv'
VAL_PATH = f'{dir_background_goal}/val_filtered.csv'
TEST_PATH = f'{dir_background_goal}/test_filtered.csv'

## 5. Métricas

#### CCC

O **CCC** (Concordance Correlation Coefficient) mede concordância entre predição e alvo,
punindo tanto erro de correlação quanto deslocamento de escala/média, por isso é sensível
ao *variance collapse* (modelo prevendo sempre perto da média).

$$\rho_c = \dfrac{2\,\rho\,\sigma_x\,\sigma_y}{\sigma_x^2 + \sigma_y^2 + (\mu_x - \mu_y)^2}$$

In [11]:
def calculate_ccc(y_true, y_pred):
    '''Concordance Correlation Coefficient sobre dois arrays 1D.'''
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true,  var_pred  = np.var(y_true),  np.var(y_pred)
    covariance = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    denominator = var_true + var_pred + (mean_true - mean_pred) ** 2
    return 0.0 if denominator == 0 else (2 * covariance) / denominator

### Perda Combinada

A função de perda usada irá combinar a BCE e a Margin Ranking Loss:

$$
Loss_{Total} = BCE_{Loss} + \lambda \cdot \text{Margin Ranking Loss}
$$

A Binary Cross Entropy vai penalizar diretamente desvios grandes na predição dos valores entre 0 e 1.

$$
BCE = - \frac{1}{N}\sum_{i=1}^{N}[y_i \cdot \log(\hat{y_i}) + (1-y_i)\cdot \log (1-\hat{y_i})]
$$

Enquanto o Margin Ranking Loss irá penalizar se o modelo atribuir baixa pontuação para amostras highlight e alta pontuação para amostras que não são highlights.

$$
Loss = \max (0, -Y \times (\hat{y}_{alto}- \hat{y}_{baixo}) + \text{margem})
$$

A margem utilizada será adaptativa, ou seja, para cada par de amostras a margem será calculada pela diferença das labels reais.

In [12]:
# CÓDIGO MORTO

"""def adaptive_margin_ranking_loss(label_low, label_high, pred_low, pred_high, margin_scale=1.0):
    margin = margin_scale * (label_high - label_low)
    return torch.relu(margin - (pred_high - pred_low))

bce = nn.BCEWithLogitsLoss()

def combined_loss(label_low, label_high, pred_low, pred_high, alpha=1.0, margin_scale=1.0):
    loss_bce_low = bce(pred_low, label_low)
    loss_bce_high = bce(pred_high, label_high)
    loss_bce = 0.5 * (loss_bce_low + loss_bce_high)

    loss_rank = adaptive_margin_ranking_loss(label_low, label_high, pred_low, pred_high, margin_scale,)

    return loss_bce + alpha * loss_rank"""

'def adaptive_margin_ranking_loss(label_low, label_high, pred_low, pred_high, margin_scale=1.0):\n    margin = margin_scale * (label_high - label_low)\n    return torch.relu(margin - (pred_high - pred_low))\n\nbce = nn.BCEWithLogitsLoss()\n\ndef combined_loss(label_low, label_high, pred_low, pred_high, alpha=1.0, margin_scale=1.0):\n    loss_bce_low = bce(pred_low, label_low)\n    loss_bce_high = bce(pred_high, label_high)\n    loss_bce = 0.5 * (loss_bce_low + loss_bce_high)\n\n    loss_rank = adaptive_margin_ranking_loss(label_low, label_high, pred_low, pred_high, margin_scale,)\n\n    return loss_bce + alpha * loss_rank'

In [13]:
class CombinedLoss(nn.Module):

    def __init__(self, alpha=1.0, margin_scale=1.0):
        super().__init__()

        self.alpha = alpha
        self.margin_scale = margin_scale
        self.bce = nn.BCEWithLogitsLoss()

    def forward(
        self,
        low_label,
        high_label,
        low_pred,
        high_pred,
        return_components=False,
    ):

        bce = (
            self.bce(low_pred, low_label)
            + self.bce(high_pred, high_label)
        ) / 2

        margin = self.margin_scale * (
            high_label - low_label
        )

        rank = torch.relu(
            margin - (high_pred - low_pred)
        ).mean()

        loss = bce + self.alpha * rank

        if return_components:
            return loss, bce, rank

        return loss

## 7. Treino

Função de treino unificada.

In [ ]:
MONITOR_MODE = {"ccc": "max", "mae": "min", "loss": "min"}

def _apply_freeze(model, freeze_backbone):
    '''Liga/desliga o gradiente da ResNet. conv1 (índice 0) fica sempre treinável.'''
    for name, param in model.cnn.named_parameters():
        param.requires_grad = name.startswith("0.") or (not freeze_backbone)

def _set_backbone_eval(model):
    '''Congela estatísticas de BatchNorm na ResNet. Conv1 fica em train.'''
    for i, module in enumerate(model.cnn):
        module.train() if i == 0 else module.eval()

def binary_accuracy(y_true, y_pred, threshold=0.5):
    return float(np.mean((y_pred > threshold) == (y_true > threshold)))

def binary_confusion_metrics(y_true, y_pred, threshold=0.5):
    pred_bin = y_pred > threshold
    target_bin = y_true > threshold

    tp = int(np.sum(pred_bin & target_bin))
    fp = int(np.sum(pred_bin & ~target_bin))
    fn = int(np.sum(~pred_bin & target_bin))
    tn = int(np.sum(~pred_bin & ~target_bin))

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": precision, "recall": recall, "f1": f1}

def _window_hit_stats(target, pred, threshold=0.5):
    '''target/pred: tensores 1D de UMA janela (window_id) — já vem assim do batch
    (cada low/high já é uma janela inteira, pois batch_size=1 + groups=window_id).'''
    target_bin = target > threshold
    pred_bin = pred > threshold
    if not target_bin.any():
        return None  # janela sem nenhum clipe "gol" de verdade, tipo janela de background
    n_true = target_bin.sum().item()
    n_hit = (pred_bin & target_bin).sum().item()
    return {
        "all_hit": n_hit == n_true,
        "any_hit": n_hit > 0,
        "hit_frac": n_hit / n_true,
    }

def _is_better(current, best, mode):
    return current > best if mode == "max" else current < best

def _log_pred_scatter(writer, y_true, y_pred, tag, step, threshold=0.5):
    '''Scatter pred x target. Colorido por TP/FP/FN/TN em relação ao threshold.'''
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    pred_bin = y_pred > threshold
    target_bin = y_true > threshold

    colors = np.select(
        [pred_bin & target_bin, pred_bin & ~target_bin, ~pred_bin & target_bin],
        ["tab:green", "tab:red", "tab:orange"],
        default="tab:gray",  # TN
    )

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(y_true, y_pred, s=8, alpha=0.4, c=colors)

    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, "--", color="gray", linewidth=1)  # diagonal ideal
    ax.axhline(threshold, color="black", linestyle=":", linewidth=0.8)
    ax.axvline(threshold, color="black", linestyle=":", linewidth=0.8)

    ax.set_xlabel("target"); ax.set_ylabel("pred"); ax.set_title("pred vs target")

    handles = [
        mpatches.Patch(color="tab:green", label="TP"),
        mpatches.Patch(color="tab:red", label="FP"),
        mpatches.Patch(color="tab:orange", label="FN"),
        mpatches.Patch(color="tab:gray", label="TN"),
    ]
    ax.legend(handles=handles, fontsize=6, loc="upper left")

    writer.add_figure(tag, fig, step)
    plt.close(fig)


def train_model(
    model, train_loader, val_loader, *,
    criterion, run_name, optimizer, scheduler,
    backbone_name, loss_name,
    freeze_mode="finetune", unfreeze_epoch=5,
    freeze_bn_always=True,
    epochs=EPOCHS, patience=PATIENCE, monitor=MONITOR,
    grad_clip=GRAD_CLIP, use_amp=False,
    checkpoint_path=None, device=device,
    trial=None,
    lr=LR,
):
    '''Treina e valida por época, logando tudo no TensorBoard.

    Args:
        criterion: função de perda (vindo de LOSSES).
        run_name: nome do experimento (usado no log_dir e no checkpoint).
        backbone_name, loss_name: usados nos hparams do TensorBoard.
        freeze_mode: "full" | "frozen" | "finetune".
        unfreeze_epoch: época de descongelamento (só vale para "finetune").
        monitor: métrica de checkpoint/early-stopping ("ccc" | "mae" | "loss").
        use_amp: ativa mixed precision (mais rápido; cheque o ccc_loss em fp16).
    Returns:
        dict com as melhores métricas e o caminho do checkpoint.
    '''
    model.to(device)
    torch.cuda.empty_cache()
    if checkpoint_path is None:
        checkpoint_path = os.path.join(CKPT_DIR, f"{run_name}.pth")
    last_path = os.path.join(CKPT_DIR, f"{run_name}_last.pth")

    mode = MONITOR_MODE[monitor]
    best_score = -np.inf if mode == "max" else np.inf
    best_metrics, best_true, best_pred = {}, None, None
    epochs_no_improve = 0

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(RUNS_DIR, f"{run_name}_{stamp}")
    writer = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    for epoch in range(epochs):
        # estado de congelamento desta época
        if freeze_mode == "full":
            backbone_frozen = False
        elif freeze_mode == "frozen":
            backbone_frozen = True
        else:  # finetune
            backbone_frozen = epoch < unfreeze_epoch
            if epoch == unfreeze_epoch:
                print(f"[época {epoch+1}] descongelando a ResNet (fine-tuning completo)")

        _apply_freeze(model, backbone_frozen)

        # ----- treino -----
        model.train()
        if freeze_bn_always or backbone_frozen:
            _set_backbone_eval(model)

        train_loss = 0.0
        train_bce = 0.0
        train_rank = 0.0

        train_true, train_pred = [], []
        
        for (low, high) in tqdm(train_loader, desc=f"época {epoch+1}/{epochs} [treino]", leave=False):
            low_video, _, low_mel, low_targets = low
            high_video, _, high_mel, high_targets = high

            low_video = low_video.to(device, non_blocking=True)
            high_video = high_video.to(device, non_blocking=True)

            low_mel = low_mel.to(device, non_blocking=True)
            high_mel = high_mel.to(device, non_blocking=True)

            low_targets = low_targets.to(device).float().view(-1)
            high_targets = high_targets.to(device).float().view(-1)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda", enabled=use_amp):
                low_outputs = model(low_video, low_mel).reshape(-1)
                high_outputs = model(high_video, high_mel).reshape(-1)

                loss, bce_loss, rank_loss = criterion(
                    low_targets,
                    high_targets,
                    low_outputs,
                    high_outputs,
                    return_components=True,
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                grad_clip,
            )

            scaler.step(optimizer)
            scaler.update()

            batch_size = low_video.size(0)
            train_loss += loss.item() * batch_size
            train_bce += bce_loss.item() * batch_size
            train_rank += rank_loss.item() * batch_size

            preds = torch.cat([torch.sigmoid(low_outputs), torch.sigmoid(high_outputs)])
            targets = torch.cat([ low_targets,  high_targets])

            train_true.extend(targets.detach().cpu().numpy())
            train_pred.extend(preds.detach().cpu().numpy())
        
        train_loss /= len(train_loader.dataset)
        train_bce /= len(train_loader.dataset)
        train_rank /= len(train_loader.dataset)

        train_true, train_pred = np.array(train_true), np.array(train_pred)
        train_mae = np.mean(np.abs(train_true - train_pred))
        train_ccc = calculate_ccc(train_true, train_pred)
        train_acc = binary_accuracy(train_true, train_pred)

        # ----- validação -----
        model.eval()

        val_loss = 0.0
        val_bce = 0.0
        val_rank = 0.0

        all_true, all_pred = [], []
        window_all_hit, window_any_hit, window_hit_frac = [], [], []

        with torch.no_grad():

            for (low, high) in tqdm(
                val_loader,
                desc=f"época {epoch+1}/{epochs} [val]",
                leave=False,
            ):

                low_video, _, low_mel, low_targets = low
                high_video, _, high_mel, high_targets = high

                low_video = low_video.to(device, non_blocking=True)
                high_video = high_video.to(device, non_blocking=True)

                low_mel = low_mel.to(device, non_blocking=True)
                high_mel = high_mel.to(device, non_blocking=True)

                low_targets = low_targets.to(device).float().view(-1)
                high_targets = high_targets.to(device).float().view(-1)

                with torch.amp.autocast("cuda", enabled=use_amp):

                    low_outputs = model(low_video, low_mel).reshape(-1)
                    high_outputs = model(high_video, high_mel).reshape(-1)

                    loss, bce_loss, rank_loss = criterion(
                        low_targets,
                        high_targets,
                        low_outputs,
                        high_outputs,
                        return_components=True,
                    )

                val_loss += loss.item() * low_video.size(0)
                val_bce += bce_loss.item() * low_video.size(0)
                val_rank += rank_loss.item() * low_video.size(0)

                low_preds_sig = torch.sigmoid(low_outputs)
                high_preds_sig = torch.sigmoid(high_outputs)

                preds = torch.cat([low_preds_sig, high_preds_sig])
                targets = torch.cat([low_targets, high_targets])

                all_true.extend(targets.detach().cpu().numpy())
                all_pred.extend(preds.detach().cpu().numpy())

                for t, p in [(low_targets, low_preds_sig), (high_targets, high_preds_sig)]:
                    stats = _window_hit_stats(t, p)
                    if stats is not None:
                        window_all_hit.append(stats["all_hit"])
                        window_any_hit.append(stats["any_hit"])
                        window_hit_frac.append(stats["hit_frac"])

        val_loss /= len(val_loader.dataset)
        val_bce /= len(val_loader.dataset)
        val_rank /= len(val_loader.dataset)

        all_true = np.array(all_true)
        all_pred = np.array(all_pred)
        
        pred_std = float(np.std(all_pred))
        target_std = float(np.std(all_true))
        val_mae = np.mean(np.abs(all_true - all_pred))
        val_ccc = calculate_ccc(all_true, all_pred)
        val_acc = binary_accuracy(all_true, all_pred)
        val_confusion = binary_confusion_metrics(all_true, all_pred)

        val_window_all_hit = float(np.mean(window_all_hit)) if window_all_hit else float("nan")
        val_window_any_hit = float(np.mean(window_any_hit)) if window_any_hit else float("nan")
        val_window_hit_frac = float(np.mean(window_hit_frac)) if window_hit_frac else float("nan")

        scheduler.step(val_loss)

        if trial is not None:
            trial.report(val_rank, epoch)
            if trial.should_prune():
                writer.close()
                raise optuna.TrialPruned()

        # ----- tensorboard -----
        writer.add_scalar("Loss/train_total", train_loss, epoch)
        writer.add_scalar("Loss/train_bce", train_bce, epoch)
        writer.add_scalar("Loss/train_rank", train_rank, epoch)

        writer.add_scalar("Loss/val_total", val_loss, epoch)
        writer.add_scalar("Loss/val_bce", val_bce, epoch)
        writer.add_scalar("Loss/val_rank", val_rank, epoch)

        writer.add_scalar("Metrics/MAE_train", train_mae, epoch)
        writer.add_scalar("Metrics/MAE_val", val_mae, epoch)
        writer.add_scalar("Metrics/CCC_train", train_ccc, epoch)
        writer.add_scalar("Metrics/CCC_val", val_ccc, epoch)
        writer.add_scalar("Metrics/Acc_train", train_acc, epoch)
        writer.add_scalar("Metrics/Acc_val", val_acc, epoch)
        writer.add_scalar("Metrics/Precision_val", val_confusion["precision"], epoch)
        writer.add_scalar("Metrics/Recall_val", val_confusion["recall"], epoch)
        writer.add_scalar("Metrics/F1_val", val_confusion["f1"], epoch)
        writer.add_scalar("Metrics/TP_val", val_confusion["tp"], epoch)
        writer.add_scalar("Metrics/FP_val", val_confusion["fp"], epoch)
        writer.add_scalar("Metrics/FN_val", val_confusion["fn"], epoch)
        writer.add_scalar("Metrics/TN_val", val_confusion["tn"], epoch)
        writer.add_scalar("Metrics/WindowAllHit_val", val_window_all_hit, epoch)
        writer.add_scalar("Metrics/WindowAnyHit_val", val_window_any_hit, epoch)
        writer.add_scalar("Metrics/WindowHitFrac_val", val_window_hit_frac, epoch)
        writer.add_scalar("Diagnostics/pred_std", pred_std, epoch)
        writer.add_scalar("Diagnostics/target_std", target_std, epoch)

        writer.add_histogram("Val/predictions", all_pred, epoch)

        for gi, pg in enumerate(optimizer.param_groups):
            writer.add_scalar(f"Train/LR_group{gi}", pg["lr"], epoch)

        print(
            f"época [{epoch+1}/{epochs}]"
            f" | train {train_loss:.4f}"
            f" (bce {train_bce:.4f}, rank {train_rank:.4f})"
            f" | val {val_loss:.4f}"
            f" (bce {val_bce:.4f}, rank {val_rank:.4f})"
            f" | LR {optimizer.param_groups[0]['lr']:.2e}"
        )

        # ----- checkpoint (best + last) + early stopping -----
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "run_name": run_name}, last_path)

        current = {"ccc": val_ccc, "mae": val_mae, "loss": val_loss}[monitor]
        if _is_better(current, best_score, mode):
            best_score = current
            best_metrics = {
            "val_loss": val_loss,
            "val_bce": val_bce,
            "val_rank": val_rank,
            "val_ccc": val_ccc,
            "val_acc": val_acc,
            **val_confusion,
            "val_window_all_hit": val_window_all_hit,
            "val_window_any_hit": val_window_any_hit,
            "val_window_hit_frac": val_window_hit_frac,
            "epoch": epoch,
        }

            best_true, best_pred = all_true, all_pred
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_metrics": best_metrics,
                "run_name": run_name,
            }, checkpoint_path)
            print(f"  novo melhor ({monitor}={best_score:.4f}) salvo em {checkpoint_path}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"early stopping (sem melhora por {patience} épocas)")
                break

    # scatter do melhor epoch — diagnóstico de variance collapse
    if best_pred is not None:
        _log_pred_scatter(writer, best_true, best_pred, "Val/pred_vs_target", best_metrics["epoch"])

    # hparams para comparar runs na aba HPARAMS (run_name="." liga aos scalars)
    writer.add_hparams(
        {"backbone": backbone_name, "loss": loss_name, "freeze_mode": freeze_mode,
         "freeze_bn_always": freeze_bn_always,
         "lr": lr, "hidden_size": MODEL_DEFAULTS["hidden_size"],
         "num_layers": MODEL_DEFAULTS["num_layers"], "frame_step": MODEL_DEFAULTS["frame_step"]},
        {
            "best/val_loss": best_metrics.get("val_loss", 0.0),
            "best/val_bce": best_metrics.get("val_bce", 0.0),
            "best/val_rank": best_metrics.get("val_rank", 0.0),
            "best/val_ccc": best_metrics.get("val_ccc", 0.0),
            "best/val_acc": best_metrics.get("val_acc", 0.0),
            "best/window_all_hit": best_metrics.get("val_window_all_hit", 0.0),
            "best/window_any_hit": best_metrics.get("val_window_any_hit", 0.0),
        },
        run_name=".",
    )
    writer.close()
    print(f"concluído. melhor {monitor}: {best_score:.4f}")
    return {"checkpoint": checkpoint_path, "best_metrics": best_metrics}

In [15]:
# nova função sugerida pelo claude

def run_experiment(
    audiomae,
    alpha=1.0,
    margin_scale=1.0,
    backbone_name="resnet18",
    freeze_mode="finetune",
    unfreeze_epoch=5,
    lr=LR,
    lr_backbone=None,
    weight_decay=WEIGHT_DECAY,
    model_kwargs=None,
    criterion=None,
    epochs=EPOCHS,
    patience=PATIENCE,
    use_amp=True,
    use_fusion=True,
    always_bn=False,
    train_loader=None,
    val_loader=None,
    trial=None,
):
    model_kwargs = {**MODEL_DEFAULTS, **(model_kwargs or {})}
    run_name = f"{backbone_name}__{freeze_mode}__fusion{use_fusion}__bn{always_bn}__trial{trial.number if trial else 'manual'}"
    print(f"\n=== {run_name} ===")

    model = ResNetLSTM_MultiModal(
        audiomae,
        backbone_name=backbone_name,
        use_fusion=use_fusion,
        **model_kwargs,
    ).to(device)

    if lr_backbone is None:
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = AdamW(
            [
                {"params": model.cnn.parameters(), "lr": lr_backbone},
                {"params": model.lstm.parameters(), "lr": lr},
                {"params": model.head.parameters(), "lr": lr},
            ],
            weight_decay=weight_decay,
        )

    scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

    if criterion is None:
        criterion = CombinedLoss(alpha=alpha, margin_scale=margin_scale)

    try:
        result = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            run_name=run_name,
            optimizer=optimizer,
            scheduler=scheduler,
            backbone_name=backbone_name,
            loss_name=criterion.__class__.__name__,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            epochs=epochs,
            patience=patience,
            use_amp=use_amp,
            freeze_bn_always=always_bn,
            lr=lr,
            trial=trial,
        )
    finally:
        # Garante limpeza mesmo se o trial lançar exceção
        del model, optimizer, scheduler, criterion
        torch.cuda.empty_cache()
        import gc; gc.collect()

    return result

In [16]:
# definindo o modelo para embedding dos mel spectrogramas
model_ae = AutoModel.from_pretrained("hance-ai/audiomae", trust_remote_code=True).to(device)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_ae = AutoModel.from_pretrained("hance-ai/audiomae", trust_remote_code=True).to(device)
GROUPS = True # estamos usando o agrupamento por janela
GROUPS_COLUMN_ID = 'window_id' # coluna do agrupamento

'''
loaders = {
    bs: (
        build_multimodal_dataloader(
            csv_path=TRAIN_PATH, pair=True, split="train", batch_size=bs, shuffle=True,
            num_workers=4, is_grayscale=True, pin_memory=True,
            groups=GROUPS, column_groups_id = GROUPS_COLUMN_ID,
            video_transform=train_video_transform, mel_transform=train_mel_transform, target_shape=TARGET_SHAPE,
            epoch_frac=FRAC_EPOCH,
        ),
        build_multimodal_dataloader(
            csv_path=VAL_PATH, pair=True, split="valid", batch_size=bs, shuffle=False,
            num_workers=4, is_grayscale=True, pin_memory=True, target_shape=TARGET_SHAPE,
            groups=GROUPS, column_groups_id = GROUPS_COLUMN_ID,
        ),
    )
    for bs in [1, 2]
}
'''

FRAME_STEP = 8   # lê 1 frame a cada 8, precisa de menos RAM por vídeo

FRAC_EPOCHS_TO_TEST = [0.05, 0.1, 0.2]

loaders = {
    (bs, frac): (
        build_multimodal_dataloader(
            csv_path=TRAIN_PATH, pair=True, split="train", batch_size=bs, shuffle=True,
            num_workers=2,            
            is_grayscale=True,
            pin_memory=False,      
            groups=GROUPS, column_groups_id=GROUPS_COLUMN_ID,
            video_transform=train_video_transform, mel_transform=train_mel_transform,
            target_shape=TARGET_SHAPE, epoch_frac=frac,
            frame_step=FRAME_STEP,
        ),
        build_multimodal_dataloader(
            csv_path=VAL_PATH, pair=True, split="valid", batch_size=bs, shuffle=False,
            num_workers=2,
            is_grayscale=True,
            pin_memory=False,
            target_shape=TARGET_SHAPE,
            groups=GROUPS, column_groups_id=GROUPS_COLUMN_ID,
            frame_step=FRAME_STEP,
        ),
    )
    for bs in [1]             
    for frac in FRAC_EPOCHS_TO_TEST
}

Dataset de Treino: 61674/64464 exemplos válidos.
6109 grupos encontrados.
Low: 53650
High: 8024

890 pares de grupos criados.

Dataset de Validação: 17111/19462 exemplos válidos.
1674 grupos encontrados.
Low: 14619
High: 2492

274 pares de grupos criados.

Dataset de Treino: 61674/64464 exemplos válidos.
6109 grupos encontrados.
Low: 53650
High: 8024

890 pares de grupos criados.

Dataset de Validação: 17111/19462 exemplos válidos.
1674 grupos encontrados.
Low: 14619
High: 2492

274 pares de grupos criados.



Corrigindo possíveis arquivos corrompidos.

In [ ]:
# 1. pega os params exatos do trial 49
study_old = optuna.load_study(study_name="multimodal_search_v5_fixed", storage="sqlite:///optuna_multimodal.db")
best = study_old.trials[49]
print(best.params, best.values)

# 2. novo estudo, focado
SEARCH_EPOCHS = 80  # mais épocas

def objective_focused(trial):
    backbone = "resnet50"
    freeze_mode = "frozen"
    use_fusion = True
    always_bn = True

    lr = trial.suggest_float("lr", best.params["lr"] / 3, best.params["lr"] * 3, log=True)
    alpha = trial.suggest_float("alpha", max(0.0, best.params["alpha"] - 0.15), min(1.0, best.params["alpha"] + 0.15))
    margin_scale = trial.suggest_float("margin_scale", max(0.5, best.params["margin_scale"] - 0.3), min(2.0, best.params["margin_scale"] + 0.3))

    frac_epoch = trial.suggest_categorical("frac_epoch", FRAC_EPOCHS_TO_TEST)
    bs = 1  # resnet50
    train_loader, val_loader = loaders[(bs, frac_epoch)]

    try:
        result = run_experiment(
            audiomae=model_ae,
            backbone_name=backbone,
            freeze_mode=freeze_mode,
            unfreeze_epoch=5,
            alpha=alpha,
            margin_scale=margin_scale,
            use_fusion=use_fusion,
            always_bn=always_bn,
            lr=lr,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=SEARCH_EPOCHS,
            trial=trial,
        )
    except optuna.TrialPruned:
        raise
    except Exception as e:
        print(f"ERRO no trial {trial.number}: {e}")
        torch.cuda.empty_cache()
        import gc; gc.collect()
        return float("inf")

    return result["best_metrics"]["val_rank"]

study_focused = optuna.create_study(
    direction="minimize",
    study_name="multimodal_search_focused_resnet50_frozen_fusion_bn",
    storage="sqlite:///optuna_multimodal.db",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.HyperbandPruner(min_resource=5, max_resource=SEARCH_EPOCHS),
)
study_focused.optimize(objective_focused, n_trials=20)

print("\n=== MELHOR ===")
print(study_focused.best_params)
print(study_focused.best_value)

[I 2026-06-30 15:53:08,617] Using an existing study with name 'multimodal_search_v5_fixed' instead of creating a new one.



=== resnet34__frozen__fusionTrue__bnTrue__trial3 ===


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /home/al.pedro.santos/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|██████████| 83.3M/83.3M [00:01<00:00, 72.1MB/s]


TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal_daug_more_games/resnet34__frozen__fusionTrue__bnTrue__trial3_20260630-155311


época [1/40] | train 0.6271 (bce 0.6004, rank 1.2947) | val 6.0320 (bce 5.7965, rank 11.4413) | LR 2.61e-04
  novo melhor (loss=6.0320) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet34__frozen__fusionTrue__bnTrue__trial3.pth


...

época [1/40] | train 0.9267 (bce 0.6184, rank 0.4011) | val 9.3257 (bce 6.2303, rank 4.0265) | LR 1.71e-05
  novo melhor (loss=9.3257) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet18__finetune__fusionTrue__bnTrue__trial150.pth


época [2/40] | train 0.9203 (bce 0.6156, rank 0.3964) | val 9.1630 (bce 6.1606, rank 3.9054) | LR 1.71e-05
  novo melhor (loss=9.1630) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet18__finetune__fusionTrue__bnTrue__trial150.pth


época [3/40] | train 0.9038 (bce 0.6137, rank 0.3774) | val 8.8690 (bce 6.0924, rank 3.6119) | LR 1.71e-05
  novo melhor (loss=8.8690) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet18__finetune__fusionTrue__bnTrue__trial150.pth


época [4/40] | train 0.8383 (bce 0.5981, rank 0.3125) | val 8.2520 (bce 5.9583, rank 2.9837) | LR 1.71e-05
  novo melhor (loss=8.2520) salvo em /mnt/storage_C4/gaussian_football/models/checkpoints/resnet18__finetune__fusionTrue__bnTrue__trial150.pth


época 5/40 [treino]:  62%|██████▏   | 28/45 [00:58<00:29,  1.75s/it]